# Diss quatrains (Phase 1b) — L40 + vLLM

Generates the AABB diss quatrains the generators are fine-tuned on: every training trait alone, 250 trait pairs and 100 trait triples, each in mild and harsh, plus disses for the 5 held-out traits (test set only) and generic fallback disses with no trait.

**The style comes from `diss_style_examples.csv`**, the team's hand-picked roasts. Each one is a two-line jab: line 1 throws the person's brag back at them ("You claim you're…", "You call yourself…"), line 2 flips it into the insult ("While…", "When…", "Then…"). A diss here is **two such jabs**, which makes an AABB quatrain.

1. **One diss per request, 4 samples.** The model writes a `RHYMES:` plan, then each line with its end word in square brackets at the front; the parser strips both.
2. **Three worked examples as earlier chat turns**, each built from two jabs in the CSV. Examples close to the requested trait are left out (e.g. no "Intelligent" examples when roasting "smart"), so the model writes new jokes instead of rewording ours, and anything too close to a CSV jab is rejected.
3. **Form checks** (`form_checks.py`): 4 lines, AABB, different A and B sounds, rare end words, leftover tags, stapled end words. Line length follows the CSV: 6–18 syllables, couplet lines within 5.
4. **Judge:** the same model first lists any line that does not make plain sense, then scores 1–5 for how hard the roast hits. A diss with an unclear line can score at most 2. Scores of 4 and 5 are kept.
5. **Dataset filters:** exact and near-duplicates within a trait combination, and a cap on how often one rhyme pair is used.

**Content rules:** no swear words or vulgar language, nothing hateful about race, religion, gender, sexuality or disability. Everything else is allowed.

**Upload next to this notebook:** `traits.json` and `form_checks.py` from `data/`, and `diss_style_examples.csv` from `data/disses/` in the repo.

**Switches:** `MODE` (`"pilot"` or `"full"`) and `THINKING` (Qwen3's thinking mode for the writing step: slower, often makes more sense; compare one pilot with it on and one with it off).

**Workflow:** pilot → read the kept **and** the judge-rejected samples → set `YIELD_GUESS` to about the measured yield → `MODE = "full"` → Restart & Run All → copy `llm_out/disses.jsonl`, `llm_out/disses_rejected.jsonl` and `llm_out/disses_meta.json` to `data/llm_out/` in the repo → `python data/prepare_disses.py filter`.

**Resuming:** the full run saves after every chunk of `CHUNK_REQUESTS` requests. If it stops for any reason, rerun the whole notebook: it reloads what is saved and only asks for the disses still missing. Keep the settings unchanged between runs; to start over, delete `llm_out/`. The pilot always starts fresh.

**Model:** Qwen/Qwen3-32B-FP8 (Apache-2.0, outputs may be used as training data).

## 1. Install

In [ ]:
%pip install -q vllm pronouncing wordfreq

## 2. Settings

In [ ]:
import os, re, csv, json, math, time, random, itertools
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"   # skip FlashInfer's sampler, which needs nvcc >= 12.8 to compile
from collections import Counter, defaultdict
from datetime import date

# ======================= THE SWITCHES =======================
MODE     = "pilot"          # "pilot" = a few trait combinations | "full" = the real run
THINKING = False            # Qwen3 thinking mode for writing (the judge never thinks)
# ============================================================
assert MODE in ("pilot", "full")

MODEL_ID = "Qwen/Qwen3-32B-FP8"
SEED     = 42

# --- sampling (Qwen3's recommended settings differ with thinking on) ---
N_CANDIDATES = 4            # samples per request; the best passing one is kept
TEMPERATURE  = 0.6 if THINKING else 0.8
TOP_P        = 0.95
MAX_TOKENS   = 3000 if THINKING else 250     # RHYMES line + 4 tagged lines is about 120 tokens
N_EXAMPLES   = 3            # worked examples per request, as earlier chat turns

# --- targets: kept disses per trait combination and per intensity (mild, harsh) ---
PER_SINGLE            = 15     # each of the 35 training traits
N_PAIRS,   PER_PAIR   = 250, 6
N_TRIPLES, PER_TRIPLE = 100, 4
PER_HELDOUT           = 15     # each of the 5 held-out traits; test set only
N_FALLBACK            = 100    # generic disses with no trait
YIELD_GUESS           = 0.3    # share of requests that give a usable diss (the first pilot measured 34%)
MAX_TRIES             = 6      # a combination gets at most MAX_TRIES x target / YIELD_GUESS requests in total
CHUNK_REQUESTS        = 1500   # results are saved after every chunk; rerun the notebook to resume
PILOT_KEEP            = {"single": 3, "pair": 3, "triple": 2, "heldout": 1, "fallback": 1}
PILOT_TARGET          = 3      # kept disses wanted per combination and intensity in the pilot

# --- form checks: line length follows the style examples; keep equal to diss_form in config.yaml ---
FORM = dict(syl_min=6, syl_max=18, pair_diff=5)

# --- judge and dataset filters ---
JUDGE_MIN      = 4             # keep scores >= this (1-5 scale)
MAX_PAIR_USES  = 15            # max times one rhyme pair (e.g. proud/crowd) may appear
NEAR_DUP_ROUGE = 0.7           # ROUGE-L F1 above this = near-duplicate

OUT_DIR = "llm_out" if MODE == "full" else "llm_pilot_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("MODE:", MODE, "| THINKING:", THINKING, "| output:", OUT_DIR)

## 3. Traits, form checks and style examples

In [ ]:
from form_checks import check, last_word, near_dup, normalize, same_sound, words_of

TRAITS  = json.load(open("traits.json", encoding="utf-8"))
TRAIN   = [t for t, v in TRAITS.items() if not v["heldout"]]
HELDOUT = [t for t, v in TRAITS.items() if v["heldout"]]
print(f"{len(TRAIN)} training traits | held out: {HELDOUT}")

# CSV trait -> (how the example prompt names it, its flip side, traits.json keys it is close to).
# Examples are never shown when roasting a close trait, so held-out traits stay unseen too.
STYLE_TRAITS = {
    "Empathetic":      ("empathetic", "makes everyone's pain about themselves", ["caring"]),
    "Good-Listener":   ("a good listener", "has nothing to say", ["patient"]),
    "Collaborative":   ("collaborative", "takes the credit and does none of the work", ["helpful"]),
    "Athletic":        ("athletic", "all muscle, no brain", ["athletic"]),
    "Approachable":    ("approachable", "needy and desperate for attention", ["outgoing", "charming"]),
    "Dependable":      ("dependable", "boring and predictable", ["responsible", "loyal", "punctual"]),
    "Detail-Oriented": ("detail-oriented", "a nitpicker who misses the big picture", ["meticulous"]),
    "Self-Motivated":  ("self-motivated", "all hustle, no direction", ["ambitious", "disciplined", "hardworking"]),
    "Resourceful":     ("resourceful", "cheap and cuts every corner", ["frugal"]),
    "Organised":       ("organised", "a control freak who gets nothing done", ["organized", "tidy"]),
    "Strong":          ("strong", "fragile under the tough act", ["athletic", "confident"]),
    "Intelligent":     ("intelligent", "a know-it-all with no common sense",
                        ["smart", "knowledgeable", "eloquent", "bookish"]),
}
SKIP_WORDS = re.compile(r"\b(ass|asses|hell|damn|crap)\b", re.I)    # examples with swearing are not shown

JABS = defaultdict(list)                  # CSV trait -> two-line jabs that rhyme
for row in csv.DictReader(open("diss_style_examples.csv", encoding="utf-8")):
    lines = [l.strip() for l in normalize(row["Diss"]).split("\n") if l.strip()]
    if len(lines) == 2 and check(lines + lines, **FORM)["reasons"] in ([], ["same_sound_AB"]) \
            and not SKIP_WORDS.search(" ".join(lines)):
        JABS[row["Positive_Traits"]].append(lines)
JAB_TOKENS = [(t, Counter(t)) for jabs in JABS.values() for j in jabs
              for t in [[w for l in j for w in words_of(l)]]]
print(f"{sum(map(len, JABS.values()))} usable jabs:", {k: len(v) for k, v in JABS.items()})

## 4. Load the model

If it runs out of memory, lower `gpu_memory_utilization` to 0.85.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, max_model_len=6144 if THINKING else 4096, gpu_memory_utilization=0.90,
          enable_prefix_caching=True, seed=SEED)
gen_params   = SamplingParams(n=N_CANDIDATES, temperature=TEMPERATURE, top_p=TOP_P,
                              max_tokens=MAX_TOKENS, repetition_penalty=1.05)
judge_params = SamplingParams(temperature=0.0, max_tokens=30)

def generate(msgs):
    # -> (list of N_CANDIDATES texts per conversation, output tokens)
    outs = llm.chat(msgs, gen_params, use_tqdm=True, chat_template_kwargs={"enable_thinking": THINKING})
    return [[c.text for c in o.outputs] for o in outs], sum(len(c.token_ids) for o in outs for c in o.outputs)

def ask(msgs):
    # one short reply per conversation, thinking off
    outs = llm.chat(msgs, judge_params, use_tqdm=True, chat_template_kwargs={"enable_thinking": False})
    return [o.outputs[0].text for o in outs]

## 5. Prompt

The prompt is short on purpose: the examples carry the style. Each worked example is two jabs from the CSV with different rhyme sounds.

In [ ]:
SYSTEM = ("You write roast-battle disses: four-line rhyming insults aimed at the person you are talking to, who "
          "asked to be roasted. You never soften a roast. Reply with a RHYMES line, then exactly 4 lines, each "
          "starting with its end word in square brackets. No title, no explanation.")
INTENSITY = {
    "mild":  "mild: teasing a friend. A real burn, but no name-calling",
    "harsh": "harsh: brutal, no mercy, name-calling allowed",
}

def describe(traits):
    return "; ".join(f'"{t}" (also said as: {", ".join(TRAITS[t]["synonyms"][:2])}), flip side: {TRAITS[t]["flip"]}'
                     for t in traits)

def user_prompt(trait_desc, intensity, n_traits):
    if n_traits == 0:
        who = ("They told you nothing about themselves. Roast them for asking a chatbot to roast them, for "
               "having nothing interesting to say about themselves, or for habits almost everyone has.")
    elif n_traits == 1:
        who = f"They say they are {trait_desc}."
    else:
        who = f"They say they are all of these: {trait_desc}. Hit every trait."
    return (
        f"Roast the person. {who}\n"
        f"Intensity: {INTENSITY[intensity]}.\n"
        "Write two jabs, each a rhyming couplet: the first line throws their brag back at them, the second line "
        "flips it into the insult. Together they make an AABB quatrain, with a different rhyme sound in each jab.\n"
        "Plain, clear words: every line must make sense on its own. No swear words or vulgar language, and "
        "nothing hateful about race, religion, gender, sexuality or disability.\n"
        "First write the RHYMES line, then the 4 lines, each starting with its end word in square brackets."
    )

def example_quatrain(style_trait, rng):
    # two jabs with different rhyme sounds that pass the form checks together
    jabs = JABS[style_trait]
    for _ in range(30):
        a, b = rng.sample(jabs, 2)
        lines = a + b
        if check(lines, **FORM)["pass"]:
            return lines
    return None

def example_turn(style_trait, rng):
    lines = example_quatrain(style_trait, rng)
    if lines is None:
        return None
    name, flip, _ = STYLE_TRAITS[style_trait]
    ends = [last_word(l) for l in lines]
    answer = (f"RHYMES: {ends[0]} / {ends[1]} ; {ends[2]} / {ends[3]}\n"
              + "\n".join(f"[{e}] {l}" for e, l in zip(ends, lines)))
    return [{"role": "user", "content": user_prompt(f'"{name}", flip side: {flip}', "harsh", 1)},
            {"role": "assistant", "content": answer}]

def build_messages(traits, intensity, rng):
    allowed = [s for s, (_, _, close) in STYLE_TRAITS.items() if not set(close) & set(traits) and len(JABS[s]) >= 2]
    msgs = [{"role": "system", "content": SYSTEM}]
    for s in rng.sample(allowed, min(len(allowed), N_EXAMPLES + 2)):     # spares, in case one has no valid pair
        turn = example_turn(s, rng)
        if turn and len(msgs) < 1 + 2 * N_EXAMPLES:
            msgs += turn
    msgs.append({"role": "user", "content": user_prompt(describe(traits), intensity, len(traits))})
    return msgs

def copies_example(lines):
    # True if either jab is a near-copy of a CSV jab
    for jab in (lines[0:2], lines[2:4]):
        toks = [w for l in jab for w in words_of(l)]
        c = Counter(toks)
        if any(near_dup(toks, c, t, ct, NEAR_DUP_ROUGE) for t, ct in JAB_TOKENS):
            return True
    return False

demo = build_messages(["smart", "funny"], "harsh", random.Random(0))
for m in demo:
    print(f"--- {m['role']}\n{m['content']}")

## 6. Parser, judge and dataset filters

The judge first names any line that does not make plain sense (made-up or misused words, a phrase only there for the rhyme), then scores how hard the roast hits. Any unclear line caps the score at 2, so nonsense cannot be kept however hard it sounds. Its calibration examples are real outputs from earlier pilots plus jabs from the CSV.

In [ ]:
THINK_RE = re.compile(r"<think>.*?</think>", re.S)
TAG_RE   = re.compile(r"^\[([^\]]*)\]\s*")

def parse(raw):
    text = normalize(THINK_RE.sub("", raw))
    plan, lines, tags = None, [], []
    for l in text.strip().splitlines():
        l = l.strip()
        if len(l) > 1 and l[0] == l[-1] == '"':
            l = l[1:-1].strip()
        if not l:
            continue
        if l.upper().startswith("RHYMES"):
            plan = l.split(":", 1)[-1].strip()
            continue
        l = re.sub(r"^(\d+[\.\):]|[-*•])\s*", "", l)       # drop numbering / bullets
        m = TAG_RE.match(l)
        tag = m.group(1).strip().lower() if m else None
        if m:
            l = l[m.end():].strip()
        if not l or l.lower().startswith(("here", "title", "sure")):
            continue
        lines.append(l); tags.append(tag)
    return plan, lines, tags

JUDGE_PROMPT = """You are checking a roast for a dataset. The person asked to be roasted and said they are: {who}. Intensity asked for: {intensity}. Rhyme and metre are checked separately; ignore them.

Step 1: list every line that does not make plain sense: a made-up or misused word, a phrase that is only there for the rhyme, or a line you cannot picture.
Step 2: score the roast:
5 = hits hard, every line is clear, both couplets are real burns about the trait
4 = a solid burn, every line is clear
3 = clear, but soft, generic or not really about the trait
2 = at least one unclear line
1 = mostly nonsense, or uses swear words or vulgar language, or is hateful about race, religion, gender, sexuality or disability

Examples:
Trait "eloquent": "You're all over the place with your big-worded talk, / it's like a lecture, not a toasting chalk. / You've got a silver tongue and a library in your mind, / but half the room is checking what's on their phone behind."
UNCLEAR LINES: 2, 4
SCORE: 2

Trait "funny": "You think every line's a punch, but it's just a bad bunch, / you're juggling hot air and your ego's a lunch. / The room echoes every "ha" with a collective groan, / you laugh so loud, you sit on your joke's cold throne."
UNCLEAR LINES: 1, 2, 4
SCORE: 2

Trait "confident": "You're confident, that's plain to see, / you always speak so bold and free. / Sometimes you talk a little too long, / but hey, at least you're rarely wrong."
UNCLEAR LINES: none
SCORE: 3

Trait "dependable": "You claim you're dependable, always on the clock, / just a boring old robot stuck in autopilot lock. / You call yourself dependable, loyal to the end, / 'cause you lack the courage to break or even bend."
UNCLEAR LINES: none
SCORE: 5

Roast:
{poem}

Reply in exactly this form:
UNCLEAR LINES: <line numbers, or none>
SCORE: <1-5>"""

def parse_judge(text):
    m = re.search(r"SCORE:\s*([1-5])", text)
    score = int(m.group(1)) if m else 0
    u = re.search(r"UNCLEAR LINES:\s*(.*)", text)
    unclear = bool(u) and not re.match(r"\s*(none|-|n/?a)?\s*$", u.group(1), re.I)
    return min(score, 2) if unclear else score

def judge(items):
    # items: [(traits, intensity, lines)] -> scores 1-5 (0 if the reply could not be read)
    if not items:
        return []
    who = lambda ts: ", ".join(f'"{t}"' for t in ts) if ts else "nothing about themselves (a generic roast)"
    outs = ask([[{"role": "user", "content": JUDGE_PROMPT.format(who=who(ts), intensity=i, poem="\n".join(ls))}]
                for ts, i, ls in items])
    return [parse_judge(o) for o in outs]

assert parse_judge("UNCLEAR LINES: none\nSCORE: 5") == 5
assert parse_judge("UNCLEAR LINES: 2\nSCORE: 5") == 2
assert parse_judge("garbage") == 0

class Corpus:
    """Kept disses plus the indexes for duplicate and rhyme-pair checks."""
    def __init__(self):
        self.keys, self.pair_uses = set(), Counter()
        self.by_group = defaultdict(list)       # trait combination -> (tokens, Counter) of kept disses

    @staticmethod
    def pairs(ends):
        return [tuple(sorted(ends[0:2])), tuple(sorted(ends[2:4]))]

    def reject_reason(self, traits, lines, ends):
        if " / ".join(l.lower() for l in lines) in self.keys:
            return "duplicate"
        if any(self.pair_uses[p] >= MAX_PAIR_USES for p in self.pairs(ends)):
            return "pair_cap"
        toks = [w for l in lines for w in words_of(l)]
        c = Counter(toks)
        if any(near_dup(toks, c, t, ct, NEAR_DUP_ROUGE) for t, ct in self.by_group[tuple(traits)]):
            return "near_duplicate"
        return None

    def add(self, traits, lines, ends):
        self.keys.add(" / ".join(l.lower() for l in lines))
        for p in self.pairs(ends):
            self.pair_uses[p] += 1
        toks = [w for l in lines for w in words_of(l)]
        self.by_group[tuple(traits)].append((toks, Counter(toks)))

def pick(cands, traits, corpus):
    ok = []
    for c in cands:
        if not c["pass"]:
            continue
        why = corpus.reject_reason(traits, c["lines"], c["ends"])
        if why:
            c["pass"] = False; c["reasons"].append(why)
        else:
            ok.append(c)
    if not ok:
        return None
    # highest judge score first, then the least used rhyme pairs
    return min(ok, key=lambda c: (-c["score"], max(corpus.pair_uses[p] for p in corpus.pairs(c["ends"]))))
print("parser and judge ready")

## 7. Run

Each chunk looks at every trait combination and intensity still short of its target, asks for `ceil(short / YIELD_GUESS)` requests for it, shuffles them, and runs up to `CHUNK_REQUESTS` of them. A request keeps at most one diss. After the chunk, the kept and rejected disses are appended to `OUT_DIR` and a line goes to `progress.jsonl`; on a rerun those are reloaded and the run carries on. A combination stops getting requests once it has had `MAX_TRIES × target / YIELD_GUESS`.

In [ ]:
def diss_targets():
    rng = random.Random(SEED)
    pairs   = rng.sample(list(itertools.combinations(TRAIN, 2)), N_PAIRS)
    triples = rng.sample(list(itertools.combinations(TRAIN, 3)), N_TRIPLES)
    combos  = ([("single", (t,), PER_SINGLE) for t in TRAIN] + [("pair", p, PER_PAIR) for p in pairs]
               + [("triple", t, PER_TRIPLE) for t in triples]
               + [("heldout", (t,), PER_HELDOUT) for t in HELDOUT] + [("fallback", (), N_FALLBACK)])
    if MODE == "pilot":
        n, picked = Counter(), []
        for kind, traits, target in combos:
            n[kind] += 1
            if n[kind] <= PILOT_KEEP[kind]:
                picked.append((kind, traits, min(target, PILOT_TARGET)))
        combos = picked
    return [{"kind": k, "traits": list(ts), "intensity": i, "target": n}
            for k, ts, n in combos for i in ("mild", "harsh")]

KEPT_FILE, REJ_FILE, LOG_FILE = (f"{OUT_DIR}/{n}" for n in ("disses.jsonl", "disses_rejected.jsonl", "progress.jsonl"))

def read_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")] if os.path.exists(path) else []

def write_jsonl(path, rows, mode="w"):
    with open(path, mode, encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

if MODE == "pilot":                                   # the pilot always starts fresh
    for f in (KEPT_FILE, REJ_FILE, LOG_FILE):
        if os.path.exists(f):
            os.remove(f)

targets = diss_targets()
target_of = {(tuple(t["traits"]), t["intensity"]): ti for ti, t in enumerate(targets)}
budget = [math.ceil(MAX_TRIES * t["target"] / YIELD_GUESS) for t in targets]

# --- resume: reload finished chunks, drop anything from a chunk that was cut off ---
done = len(read_jsonl(LOG_FILE))
saved = [r for r in read_jsonl(KEPT_FILE) if r["chunk"] < done]
write_jsonl(KEPT_FILE, saved)
write_jsonl(REJ_FILE, [r for r in read_jsonl(REJ_FILE) if r["chunk"] < done])
corpus, kept, sent = Corpus(), defaultdict(list), Counter()
for r in saved:
    key = (tuple(r["traits"]), r["intensity"])
    assert key in target_of, "The settings changed since the saved run. Restore them, or delete OUT_DIR to start over."
    kept[target_of[key]].append(r)
    corpus.add(r["traits"], r["lines"], [w for p in r["rhymes"] for w in p])
for p in read_jsonl(LOG_FILE):
    sent.update({int(k): v for k, v in p["sent"].items()})
if done:
    print(f"Resuming after chunk {done}: {len(saved):,} disses already saved")

chunk, next_id, t0 = done, len(saved), time.time()
while True:
    rng = random.Random(SEED * 100_003 + chunk)
    needed = []
    for ti, t in enumerate(targets):
        short = t["target"] - len(kept[ti])
        if short > 0:
            needed += [ti] * max(0, min(math.ceil(short / YIELD_GUESS), budget[ti] - sent[ti]))
    if not needed:
        break
    rng.shuffle(needed)
    jobs = [{"ti": ti, "messages": build_messages(targets[ti]["traits"], targets[ti]["intensity"], rng)}
            for ti in needed[:CHUNK_REQUESTS]]
    print(f"chunk {chunk}: {len(jobs)} of {len(needed)} needed requests x {N_CANDIDATES} samples")
    c0 = time.time()
    n = Counter(); reasons = Counter(); scores = Counter(); chunk_sent = Counter(j["ti"] for j in jobs)
    texts, toks = generate([j["messages"] for j in jobs])
    n.update(prompts=len(jobs), tokens=toks)
    for j, outs in zip(jobs, texts):
        j["cands"] = []
        for raw in outs:
            plan, lines, tags = parse(raw)
            c = check(lines, **FORM)
            if c["pass"] and copies_example(lines):
                c["pass"] = False; c["reasons"].append("copies_example")
            c.update(plan=plan, score=None)
            if len(lines) == 4 and len(tags) == 4:
                n.update(tag_hits=sum(t == last_word(l) for t, l in zip(tags, lines)), tag_total=4)
            j["cands"].append(c)
            n.update(cands=1, form=int(c["pass"]))
    todo = [(j, c) for j in jobs for c in j["cands"] if c["pass"]]
    t = targets
    for (j, c), s in zip(todo, judge([(t[j["ti"]]["traits"], t[j["ti"]]["intensity"], c["lines"]) for j, c in todo])):
        c["score"] = s; scores[s] += 1
        if s < JUDGE_MIN:
            c["pass"] = False; c["reasons"].append(f"judge={s}")
    new_kept, new_rej = [], []
    for j in jobs:
        tg = targets[j["ti"]]
        best = pick(j["cands"], tg["traits"], corpus)
        n["usable"] += best is not None           # counted even when the target is already full, for the yield
        if best and len(kept[j["ti"]]) >= tg["target"]:
            best["reasons"].append("over_target"); best = None
        for c in j["cands"]:
            reasons.update(c["reasons"])
            if c is not best and c["lines"]:
                new_rej.append({"kind": tg["kind"], "traits": tg["traits"], "intensity": tg["intensity"],
                                "persona": "", "lines": c["lines"], "score": c["score"],
                                "reasons": c["reasons"] or ["not_picked"], "chunk": chunk})
        if best:
            e = best["ends"]
            corpus.add(tg["traits"], best["lines"], e)
            row = {"id": next_id, "kind": tg["kind"], "traits": tg["traits"], "intensity": tg["intensity"],
                   "persona": "", "lines": best["lines"], "rhymes": [e[0:2], e[2:4]],
                   "syllables": best["syllables"], "score": best["score"], "chunk": chunk}
            kept[j["ti"]].append(row); new_kept.append(row); next_id += 1
    sent.update(chunk_sent)
    write_jsonl(KEPT_FILE, new_kept, "a")             # the progress line goes last: it marks the chunk as done
    write_jsonl(REJ_FILE, new_rej, "a")
    total = sum(map(len, kept.values()))
    write_jsonl(LOG_FILE, [{"chunk": chunk, "kept": len(new_kept), "total": total, "seconds": round(time.time() - c0, 1),
                            "counts": dict(n), "reasons": dict(reasons), "scores": {str(k): v for k, v in scores.items()},
                            "sent": {str(k): v for k, v in chunk_sent.items()}}], "a")
    print(f"  kept {len(new_kept)} | total {total:,} / {sum(t['target'] for t in targets):,} "
          f"| {time.time() - t0:.0f}s this session")
    chunk += 1
print("Done:", sum(map(len, kept.values())), "disses in", KEPT_FILE)

## 8. Stats

Totals over every chunk, including those from earlier sessions of a resumed run.

In [ ]:
diss_rows = read_jsonl(KEPT_FILE)
progress  = read_jsonl(LOG_FILE)
n, reasons, score_dist = Counter(), Counter(), Counter()
for p in progress:
    n.update(p["counts"]); reasons.update(p["reasons"])
    score_dist.update({int(k): v for k, v in p["scores"].items()})
secs, n_judged = sum(p["seconds"] for p in progress), sum(score_dist.values())
by_kind = defaultdict(lambda: [0, 0])
for ti, tg in enumerate(targets):
    by_kind[tg["kind"]][0] += len(kept[ti]); by_kind[tg["kind"]][1] += tg["target"]
short = [(tg["traits"], tg["intensity"], tg["target"] - len(kept[ti]))
         for ti, tg in enumerate(targets) if len(kept[ti]) < tg["target"]]
stats = {"kept": len(diss_rows), "wanted": sum(tg["target"] for tg in targets), "chunks": len(progress),
         "prompts": n["prompts"], "candidates": n["cands"], "form_pass": n["form"] / max(n["cands"], 1),
         "judge_pass": sum(v for s, v in score_dist.items() if s >= JUDGE_MIN) / max(n_judged, 1),
         "judge_mean": sum(s * v for s, v in score_dist.items()) / max(n_judged, 1),
         "judge_dist": dict(sorted(score_dist.items())),
         "yield_": n["usable"] / max(n["prompts"], 1), "tag_adherence": n["tag_hits"] / max(n["tag_total"], 1),
         "by_kind": {k: {"kept": a, "wanted": b} for k, (a, b) in by_kind.items()},
         "short_combinations": len(short), "reasons": dict(reasons),
         "tokens": n["tokens"], "seconds": round(secs, 1)}
tokens, n_prompts = n["tokens"], n["prompts"]
print(f"Kept {stats['kept']:,} / {stats['wanted']:,} from {n_prompts:,} requests x {N_CANDIDATES} samples "
      f"in {len(progress)} chunk(s)")
print(f"Form pass {stats['form_pass']:.0%} | judge pass {stats['judge_pass']:.0%} "
      f"(mean {stats['judge_mean']:.2f}, {stats['judge_dist']}) | lines ending on their tag {stats['tag_adherence']:.0%}")
print(f"Yield: {stats['yield_']:.0%} of requests gave a usable diss -> set YIELD_GUESS to about this")
print("By kind (kept / wanted):", {k: f"{a}/{b}" for k, (a, b) in by_kind.items()})
print(f"{len(short)} combinations still short (out of requests), e.g. {short[:5]}")
print("Top reasons:", reasons.most_common(10))
print(f"{tokens:,} output tokens in {secs:.0f}s ({tokens / max(secs, 1):,.0f} tok/s, incl. judge)")
if MODE == "pilot" and stats["yield_"] > 0:
    full_wanted = 2 * (PER_SINGLE * len(TRAIN) + N_PAIRS * PER_PAIR + N_TRIPLES * PER_TRIPLE
                       + PER_HELDOUT * len(HELDOUT) + N_FALLBACK)
    reqs = full_wanted / stats["yield_"]
    print(f"Full run: {full_wanted:,} disses -> about {reqs:,.0f} requests -> roughly "
          f"{reqs / (n_prompts / secs) / 3600:.1f} h at this speed (bigger batches run faster per request)")

## 9. Samples

Read both groups. If the judge rejects disses you would keep, lower `JUDGE_MIN` to 3; if kept ones still have lines that do not make sense, try `THINKING = True`.

In [ ]:
def show(r, head):
    print(head)
    print("   " + "\n   ".join(r["lines"]) + "\n")

sample_rng = random.Random()
print("=========== KEPT ===========\n")
for r in sample_rng.sample(diss_rows, min(10, len(diss_rows))):
    show(r, f"{r['traits'] or ['(fallback)']} | {r['intensity']} | judge {r['score']}")
judged_out = [r for r in read_jsonl(REJ_FILE) if any(x.startswith("judge=") for x in r["reasons"])]
print("=========== REJECTED BY THE JUDGE ===========\n")
for r in sample_rng.sample(judged_out, min(6, len(judged_out))):
    show(r, f"{r['traits'] or ['(fallback)']} | {r['intensity']} | judge {r['score']}")

## 10. Save the run record

The disses are already saved chunk by chunk; this writes the settings, prompts and stats. The judge-rejected disses in `disses_rejected.jsonl` can serve as "flat" negatives for 1e.

In [ ]:
meta = {"date": date.today().isoformat(), "mode": MODE, "model": MODEL_ID, "license": "Apache-2.0",
        "thinking": THINKING, "n_candidates": N_CANDIDATES, "n_examples": N_EXAMPLES, "temperature": TEMPERATURE,
        "top_p": TOP_P, "seed": SEED, "judge_min": JUDGE_MIN, "form": FORM,
        "settings": {"PER_SINGLE": PER_SINGLE, "N_PAIRS": N_PAIRS, "PER_PAIR": PER_PAIR, "N_TRIPLES": N_TRIPLES,
                     "PER_TRIPLE": PER_TRIPLE, "PER_HELDOUT": PER_HELDOUT, "N_FALLBACK": N_FALLBACK,
                     "YIELD_GUESS": YIELD_GUESS, "MAX_TRIES": MAX_TRIES, "CHUNK_REQUESTS": CHUNK_REQUESTS,
                     "MAX_PAIR_USES": MAX_PAIR_USES,
                     "NEAR_DUP_ROUGE": NEAR_DUP_ROUGE},
        "style_examples": {"file": "diss_style_examples.csv", "jabs": {k: len(v) for k, v in JABS.items()},
                           "close_traits": {k: v[2] for k, v in STYLE_TRAITS.items()}},
        "prompts": {"system": SYSTEM, "user_example": user_prompt(describe(["confident"]), "harsh", 1),
                    "judge": JUDGE_PROMPT},
        "stats": stats}
json.dump(meta, open(f"{OUT_DIR}/disses_meta.json", "w", encoding="utf-8"), indent=2)
print("Saved to", OUT_DIR)